# VSCode ↔ Colab T4 GPU 런타임 브리지

이 노트북은 **Colab 브라우저 탭에서만** 실행한다. 목적은 `generate_teacher_data.ipynb`, `train_qlora.ipynb`를 VSCode에서 열어두고, 실제 셀 실행은 이 Colab 세션의 T4 GPU에서 일어나도록 터널을 열어주는 것.

**순서 (매번 새 Colab 세션마다 반복):**
1. 이 탭 상단 `런타임` → `런타임 유형 변경` → 하드웨어 가속기 **T4 GPU**로 설정 (아래 셀을 돌리기 전에 반드시 확인할 것).
2. 아래 셀들을 순서대로 실행 (Drive 마운트 셀 포함 — 반드시 이 브라우저 탭에서 인증까지 완료할 것, 이유는 아래 참고).
3. ngrok 셀이 출력하는 URL(토큰 포함)을 복사.
4. VSCode에서 `.ipynb` 파일을 열고 오른쪽 위 **"Select Kernel"** → **"Select Another Kernel..."** → **"Existing Jupyter Server..."** → 복사한 URL 붙여넣기 (최신 Jupyter 확장에서는 `Jupyter: Specify Jupyter Server for Connections` 커맨드가 이 커널 선택 흐름으로 통합되었음. 구버전이면 그 커맨드를 써도 됨).
5. 서버 표시 이름은 아무거나 입력 → 잠시 후 뜨는 커널 목록에서 `Python 3` 선택.
6. 이 Colab 브라우저 탭은 **세션 내내 열어둔 채로 둘 것** — 닫거나 유휴 시간이 지나면 터널과 커널이 종료된다.

**참고**: ngrok 무료 계정 authtoken이 필요하다. https://dashboard.ngrok.com/get-started/your-authtoken 에서 가입 후 발급받을 것 (이메일만으로 무료 가입 가능, 카드 불필요).

In [ ]:
# GPU 타입 확인 — Tesla T4가 출력되어야 함. 아니면 1번 단계로 돌아가 런타임 유형을 바꿀 것
!nvidia-smi --query-gpu=name --format=csv

In [ ]:
# Drive를 반드시 이 브라우저 탭에서 먼저 마운트할 것.
# google.colab.drive.mount()는 Colab 프론트엔드(브라우저)와 커널 사이의 전용 JS 채널로
# 인증 팝업을 띄우기 때문에, VSCode(일반 Jupyter 클라이언트)에서 직접 호출하면 인증 채널이
# 없어 실패한다 (ValueError: mount failed). 여기서 먼저 마운트해두면 같은 VM에
# OS 레벨(FUSE)로 마운트되어, VSCode의 원격 커널에서 다시 drive.mount()를 호출해도
# 이미 마운트되어 있어 인증 없이 즉시 성공한다.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q jupyter_http_over_ws pyngrok
!jupyter serverextension enable --py jupyter_http_over_ws

In [ ]:
NGROK_AUTHTOKEN = "여기에_ngrok_authtoken_붙여넣기"  # https://dashboard.ngrok.com/get-started/your-authtoken

assert NGROK_AUTHTOKEN != "여기에_ngrok_authtoken_붙여넣기", "위 줄에 실제 authtoken을 붙여넣을 것"

import time
from pyngrok import ngrok, conf

conf.get_default().auth_token = NGROK_AUTHTOKEN

TOKEN = f"vscode-{int(time.time())}"
get_ipython().system_raw(
    f'jupyter notebook --NotebookApp.allow_origin="*" --NotebookApp.port_retries=0 '
    f'--port=8888 --no-browser --ip=0.0.0.0 --NotebookApp.token="{TOKEN}" --allow-root '
    f'> /tmp/jupyter.log 2>&1 &'
)
time.sleep(6)

tunnel = ngrok.connect(8888, "http")
print("\n아래 URL을 복사해 VSCode의 'Jupyter: Specify Jupyter Server for Connections'에 붙여넣을 것:\n")
print(f"{tunnel.public_url}/?token={TOKEN}")
print("\n(문제가 생기면 !cat /tmp/jupyter.log 로 서버 로그를 확인할 것)")

## 연결이 끊겼다면

- ngrok 무료 터널은 이 셀을 다시 실행할 때마다 URL이 바뀌므로, 재연결할 때는 위 셀을 다시 돌려 새 URL을 VSCode에 다시 지정해야 함.
- Colab 무료 티어는 유휴시간 타임아웃이 있으므로, 오랫동안 자리를 비우면 세션이 끊긴다.
- 이 브리지로 연결한 상태에서도 `generate_teacher_data.ipynb`/`train_qlora.ipynb` 안의 `drive.mount(...)`, `!git clone`, `!pip install` 셀은 그대로 실행된다 (원격 커널이 실제로 그 Colab VM 위에서 돌기 때문).